In [4]:
# EJECUTAR ESTA CELDA PRIMERO QUE NADA (Y REINICIAR EL KERNEL ANTES SI FALLÓ)
import coverage.types as ct

# Parche de compatibilidad para evitar que Numba/UMAP crasheen
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

In [6]:
import sys
import types
import math
import warnings
from pathlib import Path

# 1. Neutralizar coverage para que Numba no intente compilar con tracer roto
sys.modules['coverage'] = None
sys.modules['coverage.types'] = None

# Mock liviano de UMAP para que bertopic.plotting no dispare numba
mock_umap = types.ModuleType("umap")
mock_umap.UMAP = object
sys.modules["umap"] = mock_umap

# 2. Imports seguros
import numpy as np
import pandas as pd
import plotly.express as px
from bertopic import BERTopic

warnings.filterwarnings("ignore", category=UserWarning)

# 3. Configuración de rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_acotado"
GRAFICOS_DIR = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "graficos"
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)

# 4. Tu función de visualización intacta
def topicos_palabras(topic_model, df, topicos, ruta_html=None):
    df["topic"] = topicos
    total = len(topic_model.get_topic_info().query("Topic != -1"))

    fig = topic_model.visualize_barchart(
        top_n_topics=total,
        n_words=8,
        title="<b>Palabras representativas (Híbrido - Modelo Calibrado)</b>",
    )

    colors = px.colors.qualitative.G10
    for i, trace in enumerate(fig.data):
        trace.marker.color = colors[i % len(colors)]

    fig.update_traces(
        hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>"
    )
    fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
    fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))

    altura = max(950, math.ceil(total / 4) * 300)
    fig.update_layout(
        height=altura,
        width=1300,
        plot_bgcolor="rgba(0,0,0,0)",
        paper_bgcolor="#ffffff",
    )

    if ruta_html:
        fig.write_html(ruta_html)
        print(f"Gráfico exportado en: {ruta_html}")

    fig.show()

# 5. Carga y verificación directa de disco
print("Cargando artefactos desde disco...")
topic_model_cargado = BERTopic.load(str(OUTPUT_DIR / "modelo_acotado"))
df_cargado = pd.read_parquet(OUTPUT_DIR / "df_modelo_acotado.parquet")
embeddings_cargados = np.load(OUTPUT_DIR / "embeddings_naturales.npy")

print("\n--- Verificación de Integridad ---")
print(f"• Documentos en DataFrame:   {len(df_cargado)}")
print(f"• Matriz de embeddings:      {embeddings_cargados.shape}")

topicos_activos = topic_model_cargado.get_topic_info().query("Topic != -1")
print(f"• Cantidad de tópicos:       {len(topicos_activos)} (debe ser 22)")

distribucion = df_cargado["topic_umbral"].value_counts()
outliers = distribucion.get(-1, 0)
print(f"• Outliers residuales (-1):  {outliers} ({outliers / len(df_cargado):.1%})")
print(f"• Cobertura asignada:        {len(df_cargado) - outliers} ({(len(df_cargado) - outliers) / len(df_cargado):.1%})")

keywords_t4 = [w for w, _ in topic_model_cargado.get_topic(4)[:5]]
print(f"• Keywords Topic 4:          {', '.join(keywords_t4)}")

# 6. Renderizar gráfico interactivo
print("\nGenerando barchart interactivo...")
topicos_palabras(
    topic_model_cargado,
    df_cargado,
    df_cargado["topic_umbral"],
    ruta_html=GRAFICOS_DIR / "06_verificacion_carga_disco.html",
)

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-09 14:41:48,273 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


Cargando artefactos desde disco...

--- Verificación de Integridad ---
• Documentos en DataFrame:   3556
• Matriz de embeddings:      (3556, 384)
• Cantidad de tópicos:       22 (debe ser 22)
• Outliers residuales (-1):  534 (15.0%)
• Cobertura asignada:        3022 (85.0%)
• Keywords Topic 4:          ambiental, ambiente, crisis, proyecto, sociedad

Generando barchart interactivo...
Gráfico exportado en: C:\Trabajos\anfibia-unsam\outputs\bertopic_hibrido\graficos\06_verificacion_carga_disco.html
